# 12 CatBoost Hyperparameter Optimisierung Optuna

## Import

In [1]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.metrics import roc_auc_score

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from optuna_integration import CatBoostPruningCallback

from catboost import CatBoostClassifier

import matplotlib as mpl
import matplotlib.pyplot as plt

In [2]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframe

In [3]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [4]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]
x_full, y_full = df_raw[feat_cols], df_raw["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)],
        "full": [len(df_raw), len(x_full), len(y_full)]
    }
)

train    [476168, 476168, 476168]
val         [59522, 59522, 59522]
test        [59522, 59522, 59522]
full     [595212, 595212, 595212]
dtype: object

## Hilfsvariablen

In [5]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]
feat_cols_no_calc = [c for c in feat_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

In [6]:
def eval_modell(mod_idx, model, x_train, y_train, x_val, y_val, train_time, best_iter=None):
    """AUC auf Train und Val"""
    auc_train = roc_auc_score(y_train, model.predict_proba(x_train)[:, 1])
    auc_val = roc_auc_score(y_val, model.predict_proba(x_val)[:, 1])
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_val": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "best_iter": best_iter,
        "trainingszeit": train_time
    }

In [7]:
def mv_for_cb(df, fill_var="missing"):
    """df copy mit ersetzten nans"""
    outs = df.copy()
    for col in outs.columns:
        if outs[col].dtype.name == "category":
            if fill_var not in outs[col].cat.categories:
                outs[col] = outs[col].cat.add_categories([fill_var])
            outs[col] = outs[col].fillna(fill_var)
    return outs

In [8]:
x_train_cb = mv_for_cb(x_train)
x_val_cb = mv_for_cb(x_val)
x_test_cb = mv_for_cb(x_test)
x_full_cb = mv_for_cb(x_full)

x_train_no_calc_cb = x_train_cb[feat_cols_no_calc]
x_val_no_calc_cb = x_val_cb[feat_cols_no_calc]
x_test_no_calc_cb = x_test_cb[feat_cols_no_calc]
x_full_no_calc_cb = x_full_cb[feat_cols_no_calc]

pd.Series(
    {
        "train": [len(x_train_cb), len(x_train_no_calc_cb)],
        "val": [len(x_val_cb), len(x_val_no_calc_cb)],
        "test": [len(x_test_cb), len(x_test_no_calc_cb)],
        "full": [len(x_full_cb), len(x_full_no_calc_cb)]
    }
)

train    [476168, 476168]
val        [59522, 59522]
test       [59522, 59522]
full     [595212, 595212]
dtype: object

In [9]:
def subsample(x_set, y_set, anteil, random_state=RANDOM_STATE):
    """Subsample erzeugen"""
    if anteil >= 1.0:
        return x_set, y_set
    rng = np.random.default_rng(random_state)
    n = int(len(x_set) * anteil)
    idx_sub = rng.permutation(len(x_set))[:n]
    return x_set.iloc[idx_sub], y_set.iloc[idx_sub]

In [10]:
x_sub_75, y_sub_75 = subsample(x_train_cb, y_train, 0.75)
x_sub_no_calc_75, y_sub_no_calc_75 = subsample(x_train_no_calc_cb, y_train, 0.75)

pd.Series(
    {
        "sub": len(x_sub_75),
        "sub no calc": len(x_sub_no_calc_75),
        "anteil target": y_sub_75.mean()
    }
)

sub              357126.0000
sub no calc      357126.0000
anteil target         0.0365
dtype: float64

## Optuna Hilfe

## Suchräume

|Paramerter|Bereich|
|---|---|
|learning rate|0.01-0.3|
|depth|4-10|
|l2_leaf_reg|1-30|
|random_strength|1e3-10|
|bootstrap_type|Bayesian / Bernoulli / MVS|
|bagging_temperature|0-5|
|one_hot_max_size|2-105|
|leaf_estimation_iterations|1-10|
|auto_class_weights|none / Balanced|

In [11]:
fixed_params_plain = {
    "random_seed": RANDOM_STATE,
    "task_type": "CPU",
    "eval_metric": "AUC",
    "thread_count": -1,
    "allow_writing_files": False,
    "cat_features": cat_cols,
    "use_best_model": True,
    "early_stopping_rounds": 100,
    "loss_function": "Logloss",
    "border_count": 254,
    "verbose": 0,
    "iterations": 5000,
    "boosting_type": "Plain"
}

In [12]:
fixed_params_ordered = {
    "random_seed": RANDOM_STATE,
    "task_type": "CPU",
    "eval_metric": "AUC",
    "thread_count": -1,
    "allow_writing_files": False,
    "cat_features": cat_cols,
    "use_best_model": True,
    "early_stopping_rounds": 100,
    "loss_function": "Logloss",
    "border_count": 254,
    "verbose": 0,
    "iterations": 5000,
    "boosting_type": "Ordered"
}

In [13]:
def suchraum_params(trial):
    """Suchraum definition"""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "one_hot_max_size": trial.suggest_categorical("one_hot_max_size", [2, 10, 18, 105]),
        "leaf_estimation_iterations": trial.suggest_int("leaf_estimation_iterations", 1, 10),
        "auto_class_weights": trial.suggest_categorical("auto_class_weights", [None, "Balanced"]),
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"])
    }

    if params["bootstrap_type"] == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0.0, 5.0)
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)

    return params

## Plain mit Calc

In [15]:
def objective_plain_calc(trial):
    modell = CatBoostClassifier(
        **fixed_params_plain,
        **suchraum_params(trial)
    )
    pruning_callback = CatBoostPruningCallback(trial, "AUC")

    modell.fit(x_sub_75, y_sub_75, eval_set=[(x_val_cb, y_val)], callbacks=[pruning_callback])
    pruning_callback.check_pruned()

    return roc_auc_score(y_val, modell.predict_proba(x_val_cb)[:, 1])

In [18]:
study_plain_with_calc = optuna.create_study(
    study_name = "CatBoost_Plain_with_calc",
    storage = "sqlite:///catboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=200)
)

[I 2026-08-13 02:02:48,345] A new study created in RDB with name: CatBoost_Plain_with_calc


In [20]:
study_plain_with_calc.optimize(
    objective_plain_calc,
    n_trials = 50,
    timeout = 3*60*60,
    catch = (Exception, )
)

C:\Users\linus\AppData\Local\Temp\ipykernel_16940\1179548323.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 02:38:59,744] Trial 0 finished with value: 0.62754305297685 and parameters: {'learning_rate': 0.03574712922600244, 'depth': 10, 'l2_leaf_reg': 12.057126287443763, 'random_strength': 0.24810409748678125, 'one_hot_max_size': 105, 'leaf_estimation_iterations': 7, 'auto_class_weights': None, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.9091248360355031}. Best is trial 0 with value: 0.62754305297685.
C:\Users\linus\AppData\Local\Temp\ipykernel_16940\1179548323.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 02:40:38,241] Trial 1 finished with value: 0.63056079455906

In [21]:
pd.DataFrame({
    "best auc_val": study_plain_with_calc.best_value,
    "best gini": 2 * study_plain_with_calc.best_value - 1,
    "trials": len(study_plain_with_calc.trials),
    "pruned": len([t for t in study_plain_with_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_plain_with_calc.trials if t.state.name == "FAIL"]),
    "best params": study_plain_with_calc.best_params
})

,best auc_val,best gini,trials,pruned,fail,best params
learning_rate,0.634441,0.268881,50,14,0,0.047611
depth,0.634441,0.268881,50,14,0,7
l2_leaf_reg,0.634441,0.268881,50,14,0,8.593733
random_strength,0.634441,0.268881,50,14,0,0.011423
one_hot_max_size,0.634441,0.268881,50,14,0,105
leaf_estimation_iterations,0.634441,0.268881,50,14,0,2
auto_class_weights,0.634441,0.268881,50,14,0,Balanced
bootstrap_type,0.634441,0.268881,50,14,0,Bernoulli
subsample,0.634441,0.268881,50,14,0,0.917857


## Plain ohne Calc

In [14]:
def objective_plain_no_calc(trial):
    modell = CatBoostClassifier(
        **fixed_params_plain,
        **suchraum_params(trial)
    )
    pruning_callback = CatBoostPruningCallback(trial, "AUC")

    modell.fit(x_sub_no_calc_75, y_sub_75, eval_set=[(x_val_no_calc_cb, y_val)], callbacks=[pruning_callback])
    pruning_callback.check_pruned()

    return roc_auc_score(y_val, modell.predict_proba(x_val_no_calc_cb)[:, 1])

In [15]:
study_plain_without_calc = optuna.create_study(
    study_name = "CatBoost_Plain_without_calc",
    storage = "sqlite:///catboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=200)
)

[I 2026-08-13 07:48:13,396] Using an existing study with name 'CatBoost_Plain_without_calc' instead of creating a new one.


In [16]:
study_plain_without_calc.optimize(
    objective_plain_no_calc,
    n_trials = 50,
    timeout = 3*60*60,
    catch = (Exception, )
)

C:\Users\linus\AppData\Local\Temp\ipykernel_4008\1090979878.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 07:48:33,657] Trial 17 pruned. Trial was pruned at iteration 200.
[I 2026-08-13 07:49:11,998] Trial 18 finished with value: 0.6324996579344101 and parameters: {'learning_rate': 0.1706769518716959, 'depth': 4, 'l2_leaf_reg': 19.26564052774494, 'random_strength': 1.935391410669271, 'one_hot_max_size': 10, 'leaf_estimation_iterations': 6, 'auto_class_weights': 'Balanced', 'bootstrap_type': 'Bernoulli', 'subsample': 0.5780545957082477}. Best is trial 14 with value: 0.6340737665680749.
C:\Users\linus\AppData\Local\Temp\ipykernel_4008\1090979878.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC"

In [17]:
pd.Series({
    "best auc_val": study_plain_without_calc.best_value,
    "best gini": 2 * study_plain_without_calc.best_value - 1,
    "trials": len(study_plain_without_calc.trials),
    "pruned": len([t for t in study_plain_without_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_plain_without_calc.trials if t.state.name == "FAIL"]),
    "best params": study_plain_without_calc.best_params
})

best auc_val                                             0.634627
best gini                                                0.269254
trials                                                         67
pruned                                                         29
fail                                                            0
best params     {'learning_rate': 0.17595817710624895, 'depth'...
dtype: object

In [22]:
study_plain_without_calc.best_params

{'learning_rate': 0.17595817710624895,
 'depth': 4,
 'l2_leaf_reg': 3.9143184149527266,
 'random_strength': 0.5015455446392323,
 'one_hot_max_size': 10,
 'leaf_estimation_iterations': 2,
 'auto_class_weights': None,
 'bootstrap_type': 'Bernoulli',
 'subsample': 0.7854715288197278}

## Ordered ohne Calc

In [18]:
def objective_ordered_no_calc(trial):
    modell = CatBoostClassifier(
        **fixed_params_ordered,
        **suchraum_params(trial)
    )
    pruning_callback = CatBoostPruningCallback(trial, "AUC")

    modell.fit(x_sub_no_calc_75, y_sub_75, eval_set=[(x_val_no_calc_cb, y_val)], callbacks=[pruning_callback])
    pruning_callback.check_pruned()

    return roc_auc_score(y_val, modell.predict_proba(x_val_no_calc_cb)[:, 1])

In [19]:
study_ordered_without_calc = optuna.create_study(
    study_name = "CatBoost_Ordered_without_calc",
    storage = "sqlite:///catboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=200)
)

[I 2026-08-13 08:20:51,439] A new study created in RDB with name: CatBoost_Ordered_without_calc


In [20]:
study_ordered_without_calc.optimize(
    objective_ordered_no_calc,
    n_trials = 50,
    timeout = 3*60*60,
    catch = (Exception, )
)

C:\Users\linus\AppData\Local\Temp\ipykernel_4008\2304183874.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 08:30:04,694] Trial 0 finished with value: 0.6315642459066098 and parameters: {'learning_rate': 0.03574712922600244, 'depth': 10, 'l2_leaf_reg': 12.057126287443763, 'random_strength': 0.24810409748678125, 'one_hot_max_size': 105, 'leaf_estimation_iterations': 7, 'auto_class_weights': None, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.9091248360355031}. Best is trial 0 with value: 0.6315642459066098.
C:\Users\linus\AppData\Local\Temp\ipykernel_4008\2304183874.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 08:31:39,094] Trial 1 finished with value: 0.628582405033

In [21]:
pd.Series({
    "best auc_val": study_ordered_without_calc.best_value,
    "best gini": 2 * study_ordered_without_calc.best_value - 1,
    "trials": len(study_ordered_without_calc.trials),
    "pruned": len([t for t in study_ordered_without_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_ordered_without_calc.trials if t.state.name == "FAIL"]),
    "best params": study_ordered_without_calc.best_params
})

best auc_val                                             0.633448
best gini                                                0.266896
trials                                                         50
pruned                                                         19
fail                                                            0
best params     {'learning_rate': 0.07522925545685702, 'depth'...
dtype: object

In [23]:
study_ordered_without_calc.best_params

{'learning_rate': 0.07522925545685702,
 'depth': 9,
 'l2_leaf_reg': 21.727358099820517,
 'random_strength': 1.7703555608560753,
 'one_hot_max_size': 10,
 'leaf_estimation_iterations': 3,
 'auto_class_weights': None,
 'bootstrap_type': 'Bayesian',
 'bagging_temperature': 0.0044302304989518315}

## Load Study

In [24]:
study_plain_with_calc_loaded = optuna.load_study(
    study_name = "CatBoost_Plain_with_calc",
    storage = "sqlite:///catboost_opti.db"
)


In [25]:
study_plain_without_calc_loaded = optuna.load_study(
    study_name = "CatBoost_Plain_without_calc",
    storage = "sqlite:///catboost_opti.db"
)

In [26]:
study_ordered_without_calc_loaded = optuna.load_study(
    study_name = "CatBoost_Ordered_without_calc",
    storage = "sqlite:///catboost_opti.db"
)

## Best Params again

In [27]:
study_plain_with_calc_loaded.best_params

{'learning_rate': 0.047611121022996535,
 'depth': 7,
 'l2_leaf_reg': 8.593733349151965,
 'random_strength': 0.011423191447041188,
 'one_hot_max_size': 105,
 'leaf_estimation_iterations': 2,
 'auto_class_weights': 'Balanced',
 'bootstrap_type': 'Bernoulli',
 'subsample': 0.9178574821162729}

In [28]:
study_plain_without_calc_loaded.best_params

{'learning_rate': 0.17595817710624895,
 'depth': 4,
 'l2_leaf_reg': 3.9143184149527266,
 'random_strength': 0.5015455446392323,
 'one_hot_max_size': 10,
 'leaf_estimation_iterations': 2,
 'auto_class_weights': None,
 'bootstrap_type': 'Bernoulli',
 'subsample': 0.7854715288197278}

In [29]:
study_ordered_without_calc_loaded.best_params

{'learning_rate': 0.07522925545685702,
 'depth': 9,
 'l2_leaf_reg': 21.727358099820517,
 'random_strength': 1.7703555608560753,
 'one_hot_max_size': 10,
 'leaf_estimation_iterations': 3,
 'auto_class_weights': None,
 'bootstrap_type': 'Bayesian',
 'bagging_temperature': 0.0044302304989518315}

## 100% Train

In [30]:
results = []
train_times = {}

In [31]:
name = "L_cb_opt_01"

L_cb_opt_01 = CatBoostClassifier(
    **fixed_params_plain,
    **study_plain_with_calc_loaded.best_params
)

start = time.time()
L_cb_opt_01.fit(
    x_train_cb, y_train, 
    eval_set=(x_val_cb, y_val), 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_cb_opt_01, x_train_cb, y_train, x_val_cb, y_val, train_times[name], best_iter=L_cb_opt_01.get_best_iteration())
)

0:	test: 0.6037890	best: 0.6037890 (0)	total: 36.4ms	remaining: 3m 2s
250:	test: 0.6320888	best: 0.6323677 (245)	total: 9.32s	remaining: 2m 56s
500:	test: 0.6332904	best: 0.6336649 (478)	total: 18.5s	remaining: 2m 46s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6336873958
bestIteration = 549

Shrink model to first 550 iterations.


In [32]:
name = "L_cb_opt_02"

L_cb_opt_02 = CatBoostClassifier(
    **fixed_params_plain,
    **study_plain_without_calc_loaded.best_params
)

start = time.time()
L_cb_opt_02.fit(
    x_train_no_calc_cb, y_train, 
    eval_set=(x_val_no_calc_cb, y_val), 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_cb_opt_02, x_train_no_calc_cb, y_train, x_val_no_calc_cb, y_val, train_times[name], best_iter=L_cb_opt_02.get_best_iteration())
)

0:	test: 0.5382770	best: 0.5382770 (0)	total: 87.9ms	remaining: 7m 19s
250:	test: 0.6316706	best: 0.6316958 (244)	total: 21.1s	remaining: 6m 39s
500:	test: 0.6333197	best: 0.6335711 (459)	total: 42.2s	remaining: 6m 19s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6336871949
bestIteration = 518

Shrink model to first 519 iterations.


In [33]:
name = "L_cb_opt_03"

L_cb_opt_03 = CatBoostClassifier(
    **fixed_params_ordered,
    **study_ordered_without_calc_loaded.best_params
)

start = time.time()
L_cb_opt_03.fit(
    x_train_no_calc_cb, y_train, 
    eval_set=(x_val_no_calc_cb, y_val), 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_cb_opt_03, x_train_no_calc_cb, y_train, x_val_no_calc_cb, y_val, train_times[name], best_iter=L_cb_opt_03.get_best_iteration())
)

0:	test: 0.5613484	best: 0.5613484 (0)	total: 495ms	remaining: 41m 13s
250:	test: 0.6284516	best: 0.6284516 (246)	total: 1m 28s	remaining: 27m 49s
500:	test: 0.6295876	best: 0.6296311 (412)	total: 3m 6s	remaining: 27m 55s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6296310739
bestIteration = 412

Shrink model to first 413 iterations.


In [35]:
pd.DataFrame(results)

,model_idx,auc_train,auc_val,gini,delta_auc,best_iter,trainingszeit
0,L_cb_opt_01,0.740788,0.633687,0.267375,0.107101,549,24.047395
1,L_cb_opt_02,0.664717,0.633687,0.267374,0.031030,518,52.476140
2,L_cb_opt_03,0.655782,0.629631,0.259262,0.026151,412,192.045585


## Save Models

In [36]:
Modelle = [
    (L_cb_opt_01, "L_cb_opt_01"),
    (L_cb_opt_02, "L_cb_opt_02"),
    (L_cb_opt_03, "L_cb_opt_03")
]

for modell, name in Modelle:
    modell.save_model(f"{name}.cbm")

## Notizen
- Weil ich auf cpu trainiere setze ich border_count fix auf 254. (Soll auf GPU größeren einfluss auf laufzeit haben, als auf cpu)
- Mit log=True den suchraum um den zahlenbereich nicht linear zu halten
- bagging temp oder subsample von bootstrap type abhängig
- storage über den sqlite, kopiere aus quell, erstellt eine db. (eine db für mehrere deshalb nicht study name als name und load if exists) muss ich in die .gitignore einfügen
- direction max weil höhere auc
- TPE sampler für tree based
- medianpruner (scheint mir am meisten straight forward)
- best iters lagen meist über 200 also nehme ich das mal als warmup um den early stop etwas hinaus zu zögern
- n_trials funktioniert etwas wie iterations, ich nehme erstmal 50 wenn ich ein timelimit stelle wird das wohl kaum erreicht.
- time limit setze ich pro versuch mal auf 3 stunden. also stunden * minuten * sekunden : 3 * 60 * 60
- catch um errors zu überspringen
- pc ist im ersten durchlauf abgestürzt ich muss nochmal runnen
- Nächster Schritt: Neues Notebook mit logreg und diesen modellen auf test testen + diese modelle mit den parametern dann nochmal auf train+val trainieren und auif test testen.



## Ressourcen [Aufrufdatun 13.08.2026]:
- https://forecastegy.com/posts/catboost-hyperparameter-tuning-guide-with-optuna/
- https://catboost.ai/docs/en/references/training-parameters/output
- https://www.geeksforgeeks.org/machine-learning/catboost-cross-validation-and-hyperparameter-tuning/
- https://mljourney.com/catboost-classifier-complete-guide/
- https://catboost.ai/docs/en/concepts/parameter-tuning
- https://www.geeksforgeeks.org/machine-learning/catboost-parameters-and-hyperparameters/
- https://catboost.ai/docs/en/references/training-parameters/common
- https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier
- https://github.com/catboost/catboost/blob/master/catboost/tutorials/hyperparameters_tuning/hyperparameters_tuning.ipynb
- https://github.com/optuna/optuna-examples/blob/main/catboost/catboost_pruning.py
- https://optuna-integration.readthedocs.io/en/latest/reference/generated/optuna_integration.CatBoostPruningCallback.html
- https://optuna.readthedocs.io/en/stable/tutorial/20_recipes/001_rdb.html
- https://optuna.readthedocs.io/en/stable/reference/pruners.html
- https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html
- https://stackoverflow.com/questions/69446166/a-question-about-the-n-trials-in-optuna


